# 05 — Every metric the spec declares, against the baseline

§12 of the design spec lists what is computed; §12.1 pre-registers **one**
quantity for testing — the per-window paired difference in net LP result
against HODL — with Wilcoxon and Benjamini–Hochberg over the declared family.

This notebook runs that same test over **every** declared metric.

**Read the result as exploratory.** §12 does say the headline metric is chosen
once the first numbers are in, and it says so before any run — so picking a
different headline is inside the plan. Testing all of them and keeping whatever
wins is not. Everything here is a separate family with its own FDR control, and
the pre-registered result stands on its own in notebook 02 regardless of what
appears below.

**Needs:** `results/summary.csv` and the retained traces in `results/*.jsonl`.

> **The per-swap traces this notebook reads no longer exist.** The UU matrix run
> wrote its own traces into `results/` under byte-identical seven-field
> filenames, overwriting the arbitrage-only experiment's, and `results/` is
> gitignored (2026-08-10; see `PROGRESS.md`). `results/summary.csv`,
> `results/analysis.csv` and every already-drawn figure with its `.csv` table
> view are unaffected — what is gone is the ability to re-derive *per-swap*
> behaviour for the arb-only experiment. The cells that need traces detect this
> and skip with an explanation rather than drawing a figure from the three
> ad-hoc probes that happen to survive the filter.
>
> For the same analysis on the **UU** experiment, see
> `08-uu-flow-behaviour.ipynb`.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT))

from experiments.aggregate import AXES, BASELINE_POLICY
from experiments.metrics import per_cell_extras
from experiments.stats import benjamini_hochberg, wilcoxon_paired

def matrix_traces(directory) -> list:
    """Per-swap traces with the exact seven-field matrix name.

    The sensitivity sweeps add an eighth field and must never be pooled in: a
    prefix match once put 62k `captureShare` swaps into a pre-registered figure.
    """
    return [p for p in Path(directory).glob("*.jsonl") if len(p.stem.split("-")) == 7]


TRACES = ROOT / "results"
_found = matrix_traces(TRACES)
# A handful means the ad-hoc probes that survived the filter, not the matrix.
HAVE_TRACES = len(_found) > 1000
print(f"{len(_found)} seven-field traces under {TRACES.name}/ -> usable: {HAVE_TRACES}")
if not HAVE_TRACES:
    print(
        "\nThe arbitrage-only per-swap traces are gone. The UU matrix run wrote\n"
        "its own traces into results/ under byte-identical seven-field names\n"
        "(2026-08-10, PROGRESS.md), and results/ is gitignored, so no copy\n"
        "exists. Unaffected: results/summary.csv, results/analysis.csv, and\n"
        "every already-drawn figure with the CSV table view beside it.\n"
        "Lost: re-deriving PER-SWAP behaviour for the ARB-ONLY experiment.\n"
        "Cells needing traces are skipped below and say so.\n"
        "For the UU experiment's fee behaviour see 08-uu-flow-behaviour.ipynb."
    )

frame = pd.read_csv(ROOT / "results" / "summary.csv")
extras = (
    per_cell_extras(TRACES) if HAVE_TRACES else pd.DataFrame(columns=AXES + ["policy"])
)  # about a minute; reads every trace

# The extras cover one address mode; the axis is a control that reproduces
# exactly for every policy but MEVChargeHook, so nothing is lost.
frame = frame.merge(extras, on=AXES + ["policy"], how="left")
# The frozen arb-only summary predates the 2026-08-09 rename and carries
# `arb_profit`; anything regenerated since carries `arb_mtm`. Same quantity --
# arbitrageur inventory marked at end-of-window prices -- so resolve the name
# once instead of hardcoding either.
ARB_MTM = "arb_mtm" if "arb_mtm" in frame.columns else "arb_profit"
print(f"{len(frame)} cells, baseline = {BASELINE_POLICY}")

3 seven-field traces under results/ -> usable: False

The arbitrage-only per-swap traces are gone. The UU matrix run wrote
its own traces into results/ under byte-identical seven-field names
(2026-08-10, PROGRESS.md), and results/ is gitignored, so no copy
exists. Unaffected: results/summary.csv, results/analysis.csv, and
every already-drawn figure with the CSV table view beside it.
Lost: re-deriving PER-SWAP behaviour for the ARB-ONLY experiment.
Cells needing traces are skipped below and say so.
For the UU experiment's fee behaviour see 08-uu-flow-behaviour.ipynb.
15552 cells, baseline = MyHook@3000


## The §12 list, and where each metric lives

One item needs stating rather than computing. **LVR is `il`.** With
arbitrage-only flow the system is closed, and writing out the accounting gives

$$\text{il} = \text{hodl} - \text{principal} = \text{usd}(\text{flow}) + \text{usd}(\text{fees}) = \text{arb\_mtm} + \text{fee\_income}$$

which is the loss to arbitrage before fees — the quantity LVR names.
It is asserted below, not assumed. A counterfactual zero-fee LVR would be a
different number, but it needs the price path of candles where no swap happened
and is not recoverable from the swap log.


In [2]:
traded = frame[frame["trade_count"] > 0]
residual = (traded["il"] - (traded[ARB_MTM] + traded["fee_income"])).abs().max()
print(f"worst |il - (arb_mtm + fee_income)|: {residual:.2e}")
assert residual < 1e-6, "the closed-system identity does not hold; LVR is not il"
frame["lvr"] = frame["il"]

# Ratios per cell, but only where the denominator is large enough to carry
# one. 1.6% of traded cells have |il| below a dollar, and dividing by those
# produces values in the thousands that then dominate any mean -- the first
# version of this notebook reported a "share of LVR recovered" of 2047.
FLOOR = 100.0  # USDT
big = frame["il"].abs() >= FLOOR
volume = frame["retained_volume"].where(frame["retained_volume"] > 0)

frame["net_per_1m_volume"] = 1e6 * frame["net_result"] / volume
frame["lvr_recovered"] = (frame["fee_income"] / frame["il"]).where(big)
frame["net_per_trade"] = frame["net_result"] / frame["trade_count"].where(
    frame["trade_count"] > 0
)
print(f"cells with |il| >= {FLOOR:.0f} USDT: {int(big.sum())} of {len(frame)}")
METRICS = {
    "lp_value": "LP position value",
    "fee_income": "fee income (numeraire)",
    "fee_income_token0": "fee income, token0 leg",
    "fee_income_token1": "fee income, token1 leg",
    "il": "impermanent loss, Eq. (3)",
    "net_result": "net result vs HODL, Eq. (4)  [PRE-REGISTERED]",
    "lvr": "LVR (= il, see above)",
    "retained_volume": "retained volume",
    "trade_count": "trade count",
    "gas_units_total": "gas, total units",
    "gas_units_mean": "gas, mean units per swap",
    ARB_MTM: "arbitrageur inventory at end-of-window prices",
    "net_per_1m_volume": "net result per 1M of volume",
    "lvr_recovered": "share of LVR recovered by fees",
    "net_per_trade": "net result per trade",
}
pd.DataFrame({"metric": METRICS.keys(), "what it is": METRICS.values()})

# The trace-derived metrics do not exist when the traces do not (see the first
# cell). Drop them by presence rather than maintaining a second list that would
# drift out of step with this one.
_absent = [m for m in METRICS if m not in frame.columns]
if _absent:
    print(f"metrics unavailable without traces, dropped: {', '.join(_absent)}")
    METRICS = {k: v for k, v in METRICS.items() if k in frame.columns}


worst |il - (arb_mtm + fee_income)|: 1.10e-08
cells with |il| >= 100 USDT: 7852 of 15552
metrics unavailable without traces, dropped: fee_income_token0, fee_income_token1, gas_units_total, gas_units_mean


## The same test, metric by metric

Per-window paired difference against the baseline, two-sided Wilcoxon, then
Benjamini–Hochberg at `q = 0.05` **within each metric's family** — never pooled
across metrics, which would let a win in one borrow significance from another.


In [3]:
STRATA = ["policy", "pair", "regime", "gas_price_wei", "address_mode"]


def paired(metric: str) -> pd.DataFrame:
    """Every policy against the baseline on one metric, FDR-controlled.

    Strata with no usable difference are dropped before Benjamini-Hochberg.
    They are not failed tests, they are absent ones: the trace-derived metrics
    cover a single address mode, and on the stable pair almost every cell
    trades zero times in both arms. Leaving them in doubled `m` and made the
    threshold unreachable -- the first run of this notebook reported zero
    significant results for the per-token fee income for that reason alone.

    This is a departure from the pre-registered procedure, which fixes the
    family in advance. It is why this notebook is exploratory and why the
    pre-registered net-result family in notebook 02 is left exactly as declared.
    """
    base = frame[frame["policy"] == BASELINE_POLICY]
    others = frame[frame["policy"] != BASELINE_POLICY]
    joined = others.merge(
        base[AXES + [metric]], on=AXES, suffixes=("", "_base"), how="inner"
    )
    joined["delta"] = joined[metric] - joined[f"{metric}_base"]

    rows = []
    for keys, group in joined.groupby(STRATA, dropna=False):
        rows.append(
            {**dict(zip(STRATA, keys)), **wilcoxon_paired(group["delta"].dropna())}
        )
    out = pd.DataFrame(rows)
    out = out[out["n_effective"] > 0].reset_index(drop=True)
    out["significant_fdr"] = benjamini_hochberg(out["p"].tolist())
    out["metric"] = metric
    return out


results = pd.concat([paired(m) for m in METRICS], ignore_index=True)
results.to_csv(ROOT / "results" / "all_metrics.csv", index=False)
print(f"{len(results)} live comparisons across {len(METRICS)} metrics")
print(results.groupby("metric").size().rename("strata with data").to_string())

4612 live comparisons across 11 metrics
metric
arb_profit           428
fee_income           428
il                   428
lp_value             428
lvr                  428
lvr_recovered        396
net_per_1m_volume    396
net_per_trade        396
net_result           428
retained_volume      428
trade_count          428


## Where a dynamic policy beats the baseline

Only the hooks — the static levels are the baseline's own curve, and one of them
winning on net result means only that it priced the arbitrageur out.


In [4]:
# Higher is better for all of these except the three where a smaller number is
# the good outcome; sign them so "win" means the same thing everywhere.
LOWER_IS_BETTER = {"il", "lvr", ARB_MTM, "gas_units_total", "gas_units_mean"}

hooks = results[~results["policy"].str.startswith("MyHook@")].copy()
hooks["favourable"] = np.where(
    hooks["metric"].isin(LOWER_IS_BETTER), hooks["median"] < 0, hooks["median"] > 0
)
wins = hooks[hooks["significant_fdr"] & hooks["favourable"]]

summary = (
    hooks.assign(
        win=hooks["significant_fdr"] & hooks["favourable"],
        loss=hooks["significant_fdr"] & ~hooks["favourable"],
    )
    .groupby("metric")[["win", "loss"]]
    .sum()
    .assign(tested=hooks.groupby("metric").size())
)
summary.loc[list(METRICS)].sort_values("win", ascending=False)

,win,loss,tested
metric,,,
trade_count,106,174,310
retained_volume,96,164,310
net_per_trade,28,50,288
il,18,0,310
net_result,18,148,310
lvr,18,0,310
arb_profit,18,148,310
lp_value,16,138,310
fee_income,16,160,310


In [5]:
# Which policy wins, and where.
wins.groupby(["metric", "policy"]).size().rename("significant wins").reset_index()

,metric,policy,significant wins
0,arb_profit,BAHook,10
1,arb_profit,PegCapture,8
2,fee_income,BAHook,8
3,fee_income,PegCapture,8
4,il,MEVChargeHook,18
5,lp_value,BAHook,8
6,lp_value,PegCapture,8
7,lvr,MEVChargeHook,18
8,lvr_recovered,PegCapture,4
9,net_per_1m_volume,BAHook,2


## The comparison that carries volume with it

Net result on its own is gameable — a fee high enough to price the arbitrageur
out wins by refusing to play, which is what the top of the pre-registered table
shows. The two rows below are the same question asked so that answer cannot win:
how much of the loss to arbitrage the fee recovered, and at what volume.


In [6]:
# Aggregate as a ratio of sums, never a mean of per-cell ratios: the latter is
# dominated by cells whose denominator is near zero and is not a share of
# anything. This is the same quantity the frontier shows, in a form that reads
# without a chart.
def recovery(group):
    return pd.Series(
        {
            "volume": group["retained_volume"].mean(),
            "net": group["net_result"].mean(),
            "lvr": group["il"].mean(),
            "fees": group["fee_income"].mean(),
            "recovered": group["fee_income"].sum() / group["il"].sum(),
        }
    )


shape = traded.groupby("policy").apply(recovery, include_groups=False)
statics = shape[shape.index.str.startswith("MyHook@")].sort_values("volume")

# What the static curve recovers at the same retained volume.
shape["static_at_same_volume"] = np.interp(
    shape["volume"],
    statics["volume"],
    statics["recovered"],
    left=np.nan,
    right=np.nan,
)
shape["above_static_pp"] = 100 * (shape["recovered"] - shape["static_at_same_volume"])
shape.sort_values("recovered", ascending=False).round(3)

,volume,net,lvr,fees,recovered,static_at_same_volume,above_static_pp
policy,,,,,,,
MyHook@10000,372843.893,-2207.029,5935.468,3728.439,0.628,0.628,0.000
MyHook@6000,528756.994,-2488.148,5660.690,3172.542,0.560,0.560,0.000
PegCapture,1118640.563,-2572.442,5497.242,2924.800,0.532,0.396,13.565
BAHook,632859.719,-2665.203,5613.470,2948.267,0.525,0.529,-0.393
DAHook,813509.256,-2940.586,5577.666,2637.080,0.473,0.475,-0.202
MyHook@3000,856411.169,-2992.994,5562.228,2569.234,0.462,0.462,0.000
ABHook,842203.642,-3031.735,5576.628,2544.893,0.456,0.466,-0.983
MEVChargeHookFixed,840022.234,-3051.790,5571.856,2520.067,0.452,0.467,-1.455
MEVChargeHook,398614.166,-3128.639,5358.378,2229.739,0.416,0.617,-20.085


The static levels trade recovery against volume: the highest recovery is bought
with the smallest volume. `above_static_pp` is the only column that asks whether
a policy escapes that trade-off, and it is `NaN` wherever a policy sits outside
the range the static levels span, because there is nothing to compare against
there rather than a favourable comparison.


In [7]:
# The pre-registered result, unchanged, for the record.
pre = results[results["metric"] == "net_result"]
distinct = pre.drop_duplicates(
    subset=["policy", "pair", "regime", "gas_price_wei", "p"]
)
print(
    f"pre-registered: {int(distinct['significant_fdr'].sum())} significant of {len(distinct)} distinct"
)
ab = pre[pre["policy"] == "ABHook"]
print(
    f"ABHook litmus: median {ab['median'].median():.1f}, win rate {ab['win_rate'].mean():.3f}"
)

pre-registered: 122 significant of 214 distinct
ABHook litmus: median -14.3, win rate 0.319


## Why each policy sits where it does

Beating the static curve requires charging more exactly when the trade is
more toxic. Under arbitrage-only flow toxicity is the gap between the pool
and the reference — the arbitrageur extracts roughly $L(s-t)^2/s$ — so the
test is whether a policy's fee tracks that gap.

Fee variation that does *not* track it is noise, and convexity makes noise
slightly worse than the constant equal to its mean. So the prediction is:
distance below the static curve grows with the size of the **uncorrelated**
part of a policy's fee variation.

In [8]:
if not HAVE_TRACES:
    print("skipped: needs the arbitrage-only per-swap traces -- see the first cell")
else:
    from experiments.events import fee_vs_deviation

    pairs = fee_vs_deviation(TRACES) if HAVE_TRACES else pd.DataFrame(
        columns=["policy", "pair", "fee_bps", "deviation", "reversal", "first"]
    )


    def tracking(group):
        return pd.Series(
            {
                "swaps": len(group),
                "fee_spread_bps": group["fee_bps"].quantile(0.95)
                - group["fee_bps"].quantile(0.05),
                "corr_with_gap": group["fee_bps"].corr(
                    group["deviation"], method="spearman"
                ),
            }
        )


    track = pairs.groupby("policy").apply(tracking, include_groups=False)
    track["above_static_pp"] = shape["above_static_pp"]
    track.sort_values("above_static_pp", ascending=False).round(3)


skipped: needs the arbitrage-only per-swap traces -- see the first cell


## Why `ABHook` loses: the constant sum discounts the reversal

`ABHook` holds $f(A{\to}B) + f(B{\to}A) = K$, so every increase on one side
is a discount on the other. Under arbitrage-only flow the side that becomes
active next is decided by where the pool sits against the reference — and on
a reversal that is precisely the side just discounted.

A reversal is also where the LP is most exposed: the price came back and the
position is on the wrong side of it. So the prediction is that `ABHook`
charges *less* on reversals, while a static fee charges the same on both by
construction — the control that says the split is real and not an artefact of
which trades happen to reverse.

In [9]:
from experiments.events import fee_by_reversal

rev = fee_by_reversal(ROOT / "results")
rev = rev[~rev["first"]]  # the first trade of a run has no predecessor


def split(group):
    same, flip = group[~group["reversal"]], group[group["reversal"]]
    return pd.Series(
        {
            "swaps": len(group),
            "reversal_share": group["reversal"].mean(),
            "fee_same_dir": same["fee_bps"].mean(),
            "fee_on_reversal": flip["fee_bps"].mean(),
            "fee_gap_bps": flip["fee_bps"].mean() - same["fee_bps"].mean(),
            "deviation_same": same["deviation"].mean(),
            "deviation_on_reversal": flip["deviation"].mean(),
        }
    )


rev.groupby("policy").apply(split, include_groups=False).round(4)

,swaps,reversal_share,fee_same_dir,fee_on_reversal,fee_gap_bps,deviation_same,deviation_on_reversal
policy,,,,,,,
MyHook@3000,10.0,0.3000,30.0,30.0,0.0,0.0047,0.0043
PegDefence,892.0,0.6648,1.0,1.0,0.0,0.0009,0.0009
